In [16]:
import heapq
from collections import defaultdict
import time
from functools import total_ordering


class MemoryLogger:
    """A simple memory logger class (dummy version for Python)"""
    _instance = None
    
    @staticmethod
    def getInstance():
        if MemoryLogger._instance is None:
            MemoryLogger._instance = MemoryLogger()
        return MemoryLogger._instance
    
    def reset(self):
        self.maxMemory = 0
    
    def checkMemory(self):
        pass  # Memory logging not implemented for this translation
    
    def getMaxMemory(self):
        return self.maxMemory


class Element:
    def __init__(self, tid, iutils, rutils):
        self.tid = tid
        self.iutils = iutils
        self.rutils = rutils


class UtilityList:
    def __init__(self, item):
        self.item = item
        self.elements = []
        self.sumIutils = 0
        self.sumRutils = 0

    def addElement(self, element):
        self.elements.append(element)
        self.sumIutils += element.iutils
        self.sumRutils += element.rutils


@total_ordering
class ItemsetTKO:
    def __init__(self, itemset, item, utility):
        self.itemset = itemset
        self.item = item
        self.utility = utility

    def getItemset(self):
        return self.itemset

    def getItem(self):
        return self.item

    def __lt__(self, other):
        """Compare two ItemsetTKO objects based on their utility for sorting."""
        if not isinstance(other, ItemsetTKO):
            return NotImplemented
        return self.utility < other.utility

    def __eq__(self, other):
        """Check if two ItemsetTKO objects are equal based on their utility."""
        if not isinstance(other, ItemsetTKO):
            return NotImplemented
        return self.utility == other.utility

    def __str__(self):
        """String representation of the ItemsetTKO object."""
        temp = ",".join(map(str, self.itemset))
        temp += f",{self.item}"
        return temp


class AlgoTKO_Basic:
    def __init__(self):
        self.totalTime = 0
        self.huiCount = 0
        self.k = 0
        self.minutility = 0
        self.kItemsets = []
        self.mapItemToTWU = defaultdict(int)

    def runAlgorithm(self, input_path, output_path, k):
        MemoryLogger.getInstance().reset()
        startTimestamp = time.time()
        self.minutility = 1
        self.k = k

        # Scan the database a first time to calculate the TWU of each item.
        with open(input_path, 'r') as file:
            for line in file:
                line = line.strip()
                if line == '' or line[0] in ['#', '%', '@']:
                    continue
                
                split = line.split(":")
                items = split[0].split(" ")
                transactionUtility = int(split[1])

                for item in items:
                    item = int(item)
                    self.mapItemToTWU[item] += transactionUtility

        listItems = []
        mapItemToUtilityList = {}

        for item in self.mapItemToTWU:
            uList = UtilityList(item)
            listItems.append(uList)
            mapItemToUtilityList[item] = uList

        listItems.sort(key=lambda ul: self.mapItemToTWU[ul.item])

        # SECOND DATABASE PASS TO CONSTRUCT THE UTILITY LISTS
        # OF 1-ITEMSETS HAVING TWU >= minutil (promising items)
        with open(input_path, 'r') as file:
            tid = 0
            for line in file:
                line = line.strip()
                if line == '' or line[0] in ['#', '%', '@']:
                    continue
                
                split = line.split(":")
                items = split[0].split(" ")
                utilityValues = split[2].split(" ")

                remainingUtility = 0
                revisedTransaction = []

                for i in range(len(items)):
                    pair = (int(items[i]), int(utilityValues[i]))
                    revisedTransaction.append(pair)
                    remainingUtility += pair[1]

                revisedTransaction.sort(key=lambda x: self.compareItems(x[0], x[1]))

                for item, utility in revisedTransaction:
                    remainingUtility -= utility
                    utilityListOfItem = mapItemToUtilityList[item]
                    element = Element(tid, utility, remainingUtility)
                    utilityListOfItem.addElement(element)

                tid += 1

        MemoryLogger.getInstance().checkMemory()
        self.search([], None, listItems)
        MemoryLogger.getInstance().checkMemory()
        self.totalTime = time.time() - startTimestamp

    def search(self, prefix, pUL, ULs):
        MemoryLogger.getInstance().checkMemory()

        for i in range(len(ULs)):
            X = ULs[i]
            if X.sumIutils >= self.minutility:
                self.writeOut(prefix, X.item, X.sumIutils)

            if X.sumRutils + X.sumIutils >= self.minutility:
                exULs = []
                for j in range(i + 1, len(ULs)):
                    Y = ULs[j]
                    exULs.append(self.construct(pUL, X, Y))

                newPrefix = prefix + [X.item]
                self.search(newPrefix, X, exULs)

    def writeOut(self, prefix, item, utility):
        itemset = ItemsetTKO(prefix, item, utility)
        # Push tuple with utility first for heapq to compare based on utility
        heapq.heappush(self.kItemsets, (utility, itemset))
        
        if len(self.kItemsets) > self.k:
            if utility > self.minutility:
                while len(self.kItemsets) > self.k:
                    heapq.heappop(self.kItemsets)
                self.minutility = self.kItemsets[0][0]

    def construct(self, P, px, py):
        pxyUL = UtilityList(py.item)
        for ex in px.elements:
            ey = self.findElementWithTID(py, ex.tid)
            if ey is None:
                continue

            if P is None:
                eXY = Element(ex.tid, ex.iutils + ey.iutils, ey.rutils)
                pxyUL.addElement(eXY)
            else:
                e = self.findElementWithTID(P, ex.tid)
                if e is not None:
                    eXY = Element(ex.tid, ex.iutils + ey.iutils - e.iutils, ey.rutils)
                    pxyUL.addElement(eXY)

        return pxyUL

    def findElementWithTID(self, ulist, tid):
        first = 0
        last = len(ulist.elements) - 1

        while first <= last:
            middle = (first + last) // 2
            if ulist.elements[middle].tid < tid:
                first = middle + 1
            elif ulist.elements[middle].tid > tid:
                last = middle - 1
            else:
                return ulist.elements[middle]
        return None

    def writeResultToFile(self, path):
        with open(path, 'w') as writer:
            for _, itemset in self.kItemsets:
                buffer = ' '.join(map(str, itemset.getItemset())) + f" {itemset.item} #UTIL: {itemset.utility}\n"
                writer.write(buffer)

    def compareItems(self, item1, item2):
        compare = self.mapItemToTWU[item1] - self.mapItemToTWU[item2]
        return item1 - item2 if compare == 0 else compare

    def printStats(self):
        print("=============  TKO-BASIC - Python Version =============")
        print(f"High-utility itemsets count: {len(self.kItemsets)}")
        print(f"Total time ~ {self.totalTime:.2f} s")
        print(f"Memory ~ {MemoryLogger.getInstance().getMaxMemory()} MB")
        print("===================================================")


def main():
    # Input file path
    input_file = "C:\\Users\\ngchn\\Downloads\\eihi\\DB_Utility.txt"
    
    # Output file path
    output_file = "output.txt"
    
    # The parameter k
    k = 8
    
    # Applying the algorithm
    algorithm = AlgoTKO_Basic()
    algorithm.runAlgorithm(input_file, output_file, k)
    algorithm.writeResultToFile(output_file)
    
    # Print statistics about the algorithm execution
    algorithm.printStats()

if __name__ == "__main__":
    main()


=============  TKO-BASIC - Python Version =============
High-utility itemsets count: 8
Total time ~ 0.00 s
Memory ~ 0 MB


In [17]:
import os
import psutil

class MemoryLogger:
    # The only instance of this class (this is the "singleton" design pattern)
    _instance = None

    def __init__(self):
        # Variable to store the maximum memory usage
        self.maxMemory = 0
        # A boolean flag to indicate whether the recording mode is on or off
        self.recordingMode = False
        # A file object to store the output file name
        self.outputFile = None
        # A writer object to write the memory usage values to the file
        self.writer = None

    @classmethod
    def getInstance(cls):
        if cls._instance is None:
            cls._instance = MemoryLogger()
        return cls._instance

    def getMaxMemory(self):
        """
        To get the maximum amount of memory used until now
        :return: a float value indicating memory as megabytes
        """
        return self.maxMemory

    def reset(self):
        """
        Reset the maximum amount of memory recorded.
        """
        self.maxMemory = 0

    def checkMemory(self):
        """
        Check the current memory usage and record it if it is higher than the amount
        of memory previously recorded.
        :return: the memory usage in megabytes
        """
        currentMemory = psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024
        if currentMemory > self.maxMemory:
            self.maxMemory = currentMemory
        # If the recording mode is on
        if self.recordingMode:
            # Try to write the current memory value to the file
            try:
                self.writer.write(f"{currentMemory}\n")
            except Exception as e:
                # Handle the exception
                print(e)
        return currentMemory

    def startRecordingMode(self, fileName):
        """
        A method to set the recording mode and the output file name
        :param fileName: the path to a file for saving recorded values
        """
        # Set the recording mode flag
        self.recordingMode = True
        # Create a new file object with the given file name
        self.outputFile = fileName
        # Try to create a new writer object with the file object
        try:
            self.writer = open(self.outputFile, 'w')
        except Exception as e:
            # Handle the exception
            print(e)

    def stopRecordingMode(self):
        """
        A method to stop the recording mode and close the file
        """
        # If the recording mode is on
        if self.recordingMode:
            # Try to close the writer object
            try:
                self.writer.close()
            except Exception as e:
                # Handle the exception
                print(e)
            # Set the recording mode flag to false
            self.recordingMode = False


In [18]:
class Element:
    """
    This class represents an Element of a utility list as used by the HUI-Miner algorithm.
    """

    def __init__(self, tid, iutils, rutils):
        """
        Constructor for the Element class.
        :param tid: the transaction id
        :param iutils: the itemset utility
        :param rutils: the remaining utility
        """
        self.tid = tid  # transaction id
        self.iutils = iutils  # itemset utility
        self.rutils = rutils  # remaining utility

In [19]:
class UtilityList:
    def __init__(self, item):
        """
        Constructor for the UtilityList.
        :param item: the item that is used for this utility list
        """
        self.item = item  # the item
        self.sumIutils = 0  # the sum of item utilities
        self.sumRutils = 0  # the sum of remaining utilities
        self.elements = []  # the list of elements

    def addElement(self, element):
        """
        Method to add an element to this utility list and update the sums at the same time.
        :param element: an Element object
        """
        self.sumIutils += element.iutils
        self.sumRutils += element.rutils
        self.elements.append(element)

    def getSupport(self):
        """
        Get the support of the itemset represented by this utility-list.
        :return: the support as a number of transactions
        """
        return len(self.elements)

    def getUtils(self):
        """
        Get the sum of iutil values.
        :return: the sum of item utilities
        """
        return self.sumIutils


In [20]:
import os
import time
import heapq
from collections import defaultdict

class Element:
    def __init__(self, tid, iutils, rutils):
        self.tid = tid  # transaction id
        self.iutils = iutils  # itemset utility
        self.rutils = rutils  # remaining utility

class UtilityList:
    def __init__(self, item):
        self.item = item  # the item
        self.sumIutils = 0  # the sum of item utilities
        self.sumRutils = 0  # the sum of remaining utilities
        self.elements = []  # the elements

    def addElement(self, element):
        self.sumIutils += element.iutils
        self.sumRutils += element.rutils
        self.elements.append(element)

class Pair:
    def __init__(self):
        self.item = 0
        self.utility = 0

class ItemsetTKO:
    def __init__(self, prefix, item, utility):
        self.itemset = prefix
        self.item = item
        self.utility = utility

    def __lt__(self, other):
        return self.utility < other.utility

class MemoryLogger:
    _instance = None
    maxMemory = 0

    def __init__(self):
        pass

    @staticmethod
    def getInstance():
        if MemoryLogger._instance is None:
            MemoryLogger._instance = MemoryLogger()
        return MemoryLogger._instance

    def reset(self):
        MemoryLogger.maxMemory = 0

    def checkMemory(self):
        # Simulating memory check
        currentMemory = self.getMemoryUsage()
        if currentMemory > MemoryLogger.maxMemory:
            MemoryLogger.maxMemory = currentMemory

    def getMaxMemory(self):
        return MemoryLogger.maxMemory

    def getMemoryUsage(self):
        return 0  # Simulate memory usage

class AlgoTKO_Basic:
    def __init__(self):
        self.totalTime = 0
        self.huiCount = 0
        self.k = 0
        self.minutility = 0
        self.kItemsets = []
        self.mapItemToTWU = defaultdict(int)

    def runAlgorithm(self, input, output, k):
        MemoryLogger.getInstance().reset()
        startTimestamp = time.time()
        self.minutility = 1
        self.k = k

        self.kItemsets = []

        # First pass: calculate TWU of each item
        with open(input, 'r') as myInput:
            for thisLine in myInput:
                if thisLine.strip() == '' or thisLine[0] in ('#', '%', '@'):
                    continue

                split = thisLine.split(":")
                items = split[0].split(" ")
                transactionUtility = int(split[1])
                
                for item in items:
                    self.mapItemToTWU[int(item)] += transactionUtility

        listItems = []
        mapItemToUtilityList = {}

        for item in self.mapItemToTWU.keys():
            uList = UtilityList(item)
            listItems.append(uList)
            mapItemToUtilityList[item] = uList

        listItems.sort(key=lambda x: (self.mapItemToTWU[x.item], x.item))

        # Second pass: construct utility lists
        with open(input, 'r') as myInput:
            tid = 0
            for thisLine in myInput:
                if thisLine.strip() == '' or thisLine[0] in ('#', '%', '@'):
                    continue

                split = thisLine.split(":")
                items = split[0].split(" ")
                utilityValues = split[2].split(" ")

                remainingUtility = 0
                revisedTransaction = []

                for i in range(len(items)):
                    pair = Pair()
                    pair.item = int(items[i])
                    pair.utility = int(utilityValues[i])
                    revisedTransaction.append(pair)
                    remainingUtility += pair.utility

                revisedTransaction.sort(key=lambda x: x.item)

                for pair in revisedTransaction:
                    remainingUtility -= pair.utility
                    utilityListOfItem = mapItemToUtilityList[pair.item]
                    element = Element(tid, pair.utility, remainingUtility)
                    utilityListOfItem.addElement(element)

                tid += 1

        MemoryLogger.getInstance().checkMemory()

        self.search([], None, listItems)

        MemoryLogger.getInstance().checkMemory()
        self.totalTime = time.time() - startTimestamp

    def search(self, prefix, pUL, ULs):
        MemoryLogger.getInstance().checkMemory()

        for i in range(len(ULs)):
            X = ULs[i]

            if X.sumIutils >= self.minutility:
                self.writeOut(prefix, X.item, X.sumIutils)

            if X.sumRutils + X.sumIutils >= self.minutility:
                exULs = []
                for j in range(i + 1, len(ULs)):
                    Y = ULs[j]
                    exULs.append(self.construct(pUL, X, Y))

                newPrefix = prefix + [X.item]
                self.search(newPrefix, X, exULs)

    def writeOut(self, prefix, item, utility):
        itemset = ItemsetTKO(prefix, item, utility)
        heapq.heappush(self.kItemsets, itemset)
        if len(self.kItemsets) > self.k:
            heapq.heappop(self.kItemsets)
            self.minutility = self.kItemsets[0].utility

    def construct(self, P, px, py):
        pxyUL = UtilityList(py.item)
        for ex in px.elements:
            ey = self.findElementWithTID(py, ex.tid)
            if ey is None:
                continue
            if P is None:
                eXY = Element(ex.tid, ex.iutils + ey.iutils, ey.rutils)
                pxyUL.addElement(eXY)
            else:
                e = self.findElementWithTID(P, ex.tid)
                if e:
                    eXY = Element(ex.tid, ex.iutils + ey.iutils - e.iutils, ey.rutils)
                    pxyUL.addElement(eXY)
        return pxyUL

    def findElementWithTID(self, ulist, tid):
        first = 0
        last = len(ulist.elements) - 1

        while first <= last:
            middle = (first + last) // 2
            if ulist.elements[middle].tid < tid:
                first = middle + 1
            elif ulist.elements[middle].tid > tid:
                last = middle - 1
            else:
                return ulist.elements[middle]
        return None

    def writeResultTofile(self, path):
        with open(path, 'w') as writer:
            while self.kItemsets:
                itemset = heapq.heappop(self.kItemsets)
                writer.write(' '.join(map(str, itemset.itemset + [itemset.item])) + f" #UTIL: {itemset.utility}\n")

    def printStats(self):
        print("=============  TKO-BASIC - v.2.28 =============")
        print(f" High-utility itemsets count : {len(self.kItemsets)}")
        print(f" Total time ~ {self.totalTime} s")
        print(f" Memory ~ {MemoryLogger.getInstance().getMaxMemory()} MB")
        print("===================================================")


In [21]:
import os
import urllib.parse

class MainTestTKOBasic:

    @staticmethod
    def main():
        # input file path
        input_file = MainTestTKOBasic.fileToPath("db_utility_filtered_3.txt")
        
        # output file path
        output = "outputTKO.txt"
        
        # the parameter k
        k = 3
        # Applying the algorithm
        algorithm = AlgoTKO_Basic()
        algorithm.runAlgorithm(input_file, output, k)
        algorithm.writeResultTofile(output)
        
        # Print statistics about the algorithm execution
        algorithm.printStats()

    @staticmethod
    def fileToPath(filename):
        # Since __file__ is not available, use the current working directory
        path = os.path.join(os.getcwd(), filename)
        return urllib.parse.unquote(path)

if __name__ == "__main__":
    MainTestTKOBasic.main()


=============  TKO-BASIC - v.2.28 =============
 High-utility itemsets count : 0
 Total time ~ 11.905390501022339 s
 Memory ~ 0 MB
